 Proyecto de Ciencia de Datos
Integrantes: Otto Ferrer y Gema Zambrano

Librerias

In [57]:

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

Cargado de base de datos

In [58]:
#1. Carga: etiquetas binarias (0/1) generadas por VisualCheXbert sobre CheXpert.
url2 = "https://github.com/ottoferrer/patient_data/raw/main/train_visualCheXbert.csv"
dataset = pd.read_csv(url2)
dataset = pd.DataFrame(dataset)
dataset.head()

,Path,Sex,Age,Frontal/Lateral,AP/PA,Enlarged Cardiomediastinum,Cardiomegaly,Lung Opacity,Lung Lesion,Edema,Consolidation,Pneumonia,Atelectasis,Pneumothorax,Pleural Effusion,Pleural Other,Fracture,Support Devices,No Finding
0,CheXpert-v1.0/train/patient00001/study1/view1_...,Female,68,Frontal,AP,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1,CheXpert-v1.0/train/patient00002/study2/view1_...,Female,87,Frontal,AP,1.0,1.0,1.0,0.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0
2,CheXpert-v1.0/train/patient00002/study1/view1_...,Female,83,Frontal,AP,1.0,1.0,1.0,1.0,0.0,1.0,1.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0
3,CheXpert-v1.0/train/patient00002/study1/view2_...,Female,83,Lateral,NaN,1.0,1.0,1.0,1.0,0.0,1.0,1.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0
4,CheXpert-v1.0/train/patient00003/study1/view1_...,Male,41,Frontal,AP,1.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [59]:
#2. Carga: datos demográficos de los pacientes en CheXpert.
data_dir = '/Users/gemazambrano/Downloads/chexpertdemodata-2/'
dataset_demo= pd.DataFrame(pd.read_excel(data_dir + 'CHEXPERT DEMO.xlsx', engine='openpyxl'))
dataset_demo.head()


,PATIENT,GENDER,AGE_AT_CXR,PRIMARY_RACE,ETHNICITY
0,patient24428,Male,61,White,Non-Hispanic/Non-Latino
1,patient48289,Female,39,Other,Hispanic/Latino
2,patient33856,Female,81,White,Non-Hispanic/Non-Latino
3,patient41673,Female,42,Unknown,Unknown
4,patient48493,Male,71,White,Non-Hispanic/Non-Latino


Limpieza de las bases de datos

In [60]:
# dataset contiene informacion en la columna 'Path' separar por cada '/' y obtener cada parte en una nueva columna'
dataset[['version','split','Patient_id','Study','Image']] = dataset['Path'].str.split('/', expand=True)
dataset.head()

,Path,Sex,Age,Frontal/Lateral,AP/PA,Enlarged Cardiomediastinum,Cardiomegaly,Lung Opacity,Lung Lesion,Edema,...,Pleural Effusion,Pleural Other,Fracture,Support Devices,No Finding,version,split,Patient_id,Study,Image
0,CheXpert-v1.0/train/patient00001/study1/view1_...,Female,68,Frontal,AP,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,CheXpert-v1.0,train,patient00001,study1,view1_frontal.jpg
1,CheXpert-v1.0/train/patient00002/study2/view1_...,Female,87,Frontal,AP,1.0,1.0,1.0,0.0,1.0,...,1.0,0.0,0.0,0.0,0.0,CheXpert-v1.0,train,patient00002,study2,view1_frontal.jpg
2,CheXpert-v1.0/train/patient00002/study1/view1_...,Female,83,Frontal,AP,1.0,1.0,1.0,1.0,0.0,...,0.0,1.0,1.0,0.0,0.0,CheXpert-v1.0,train,patient00002,study1,view1_frontal.jpg
3,CheXpert-v1.0/train/patient00002/study1/view2_...,Female,83,Lateral,NaN,1.0,1.0,1.0,1.0,0.0,...,0.0,1.0,1.0,0.0,0.0,CheXpert-v1.0,train,patient00002,study1,view2_lateral.jpg
4,CheXpert-v1.0/train/patient00003/study1/view1_...,Male,41,Frontal,AP,1.0,1.0,1.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,CheXpert-v1.0,train,patient00003,study1,view1_frontal.jpg


In [61]:
#Se estan renombrando las columnas del dataset demografico para que coincidan con el dataset principal
dataset_demo = dataset_demo.rename(columns={'PRIMARY_RACE': 'Race'})
dataset_demo = dataset_demo.rename(columns={'PATIENT': 'Patient_id'})
dataset_demo = dataset_demo.rename(columns={'GENDER': 'Sex'})
dataset_demo = dataset_demo.rename(columns={'AGE_AT_CXR': 'Age'})
dataset_demo = dataset_demo.rename(columns={'ETHNICITY': 'Ethnicity'})
dataset_demo.head()



,Patient_id,Sex,Age,Race,Ethnicity
0,patient24428,Male,61,White,Non-Hispanic/Non-Latino
1,patient48289,Female,39,Other,Hispanic/Latino
2,patient33856,Female,81,White,Non-Hispanic/Non-Latino
3,patient41673,Female,42,Unknown,Unknown
4,patient48493,Male,71,White,Non-Hispanic/Non-Latino


In [62]:
#creacion del nuevo dataset uniendo ambos datasets por la columna 'Patient_id'
dataset_demo_l = dataset_demo.drop(['Sex', 'Age'], axis=1)
final_dataset = pd.merge(dataset, dataset_demo_l, on='Patient_id', how='left')
final_dataset.head()

,Path,Sex,Age,Frontal/Lateral,AP/PA,Enlarged Cardiomediastinum,Cardiomegaly,Lung Opacity,Lung Lesion,Edema,...,Fracture,Support Devices,No Finding,version,split,Patient_id,Study,Image,Race,Ethnicity
0,CheXpert-v1.0/train/patient00001/study1/view1_...,Female,68,Frontal,AP,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,CheXpert-v1.0,train,patient00001,study1,view1_frontal.jpg,Other,Non-Hispanic/Non-Latino
1,CheXpert-v1.0/train/patient00002/study2/view1_...,Female,87,Frontal,AP,1.0,1.0,1.0,0.0,1.0,...,0.0,0.0,0.0,CheXpert-v1.0,train,patient00002,study2,view1_frontal.jpg,"White, non-Hispanic",Non-Hispanic/Non-Latino
2,CheXpert-v1.0/train/patient00002/study1/view1_...,Female,83,Frontal,AP,1.0,1.0,1.0,1.0,0.0,...,1.0,0.0,0.0,CheXpert-v1.0,train,patient00002,study1,view1_frontal.jpg,"White, non-Hispanic",Non-Hispanic/Non-Latino
3,CheXpert-v1.0/train/patient00002/study1/view2_...,Female,83,Lateral,NaN,1.0,1.0,1.0,1.0,0.0,...,1.0,0.0,0.0,CheXpert-v1.0,train,patient00002,study1,view2_lateral.jpg,"White, non-Hispanic",Non-Hispanic/Non-Latino
4,CheXpert-v1.0/train/patient00003/study1/view1_...,Male,41,Frontal,AP,1.0,1.0,1.0,0.0,1.0,...,0.0,0.0,0.0,CheXpert-v1.0,train,patient00003,study1,view1_frontal.jpg,"White, non-Hispanic",Non-Hispanic/Non-Latino


In [63]:
#Revisión de valores nulos en el nuevo dataset
final_dataset.isnull().sum()
#seleccionar solo las filas sin valores nulos
final_dataset_clean = final_dataset.dropna()
final_dataset_clean.isnull().sum()
final_dataset_clean.shape
final_dataset_clean.head()


,Path,Sex,Age,Frontal/Lateral,AP/PA,Enlarged Cardiomediastinum,Cardiomegaly,Lung Opacity,Lung Lesion,Edema,...,Fracture,Support Devices,No Finding,version,split,Patient_id,Study,Image,Race,Ethnicity
0,CheXpert-v1.0/train/patient00001/study1/view1_...,Female,68,Frontal,AP,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,CheXpert-v1.0,train,patient00001,study1,view1_frontal.jpg,Other,Non-Hispanic/Non-Latino
1,CheXpert-v1.0/train/patient00002/study2/view1_...,Female,87,Frontal,AP,1.0,1.0,1.0,0.0,1.0,...,0.0,0.0,0.0,CheXpert-v1.0,train,patient00002,study2,view1_frontal.jpg,"White, non-Hispanic",Non-Hispanic/Non-Latino
2,CheXpert-v1.0/train/patient00002/study1/view1_...,Female,83,Frontal,AP,1.0,1.0,1.0,1.0,0.0,...,1.0,0.0,0.0,CheXpert-v1.0,train,patient00002,study1,view1_frontal.jpg,"White, non-Hispanic",Non-Hispanic/Non-Latino
4,CheXpert-v1.0/train/patient00003/study1/view1_...,Male,41,Frontal,AP,1.0,1.0,1.0,0.0,1.0,...,0.0,0.0,0.0,CheXpert-v1.0,train,patient00003,study1,view1_frontal.jpg,"White, non-Hispanic",Non-Hispanic/Non-Latino
5,CheXpert-v1.0/train/patient00004/study1/view1_...,Female,20,Frontal,PA,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,CheXpert-v1.0,train,patient00004,study1,view1_frontal.jpg,Black or African American,Non-Hispanic/Non-Latino


Revision de las variables categoricas del set de datos

In [64]:
# Revisión de la variable 'Race'
print(final_dataset_clean['Race'].value_counts())

# Estandarizacion de categorias de la variable 'Race'

final_dataset_clean["race_standardized"] = np.nan

# Se normalizan los datos en minúsculas y se eliminan espacios en blanco
final_dataset_clean["race_raw_clean"] = (final_dataset_clean["Race"].str.lower().str.strip())

# White
final_dataset_clean.loc[final_dataset_clean["race_raw_clean"].str.contains("white|caucasian", na=False),
       "race_standardized"] = "White"

# Black or African American
final_dataset_clean.loc[final_dataset_clean["race_raw_clean"].str.contains("black|african", na=False),
       "race_standardized"] = "Black or African American"

# Asian
final_dataset_clean.loc[final_dataset_clean["race_raw_clean"].str.contains("asian", na=False),
       "race_standardized"] = "Asian"

# American Indian or Alaska Native
final_dataset_clean.loc[final_dataset_clean["race_raw_clean"].str.contains("american indian|native american|alaska", na=False),
       "race_standardized"] = "American Indian or Alaska Native"

# Native Hawaiian or Other Pacific Islander
final_dataset_clean.loc[final_dataset_clean["race_raw_clean"].str.contains("hawaiian|pacific islander", na=False),
       "race_standardized"] = "Native Hawaiian or Other Pacific Islander"

# Other
final_dataset_clean.loc[final_dataset_clean["race_raw_clean"].str.contains("^other| other,", regex=True, na=False),
       "race_standardized"] = "Other"

# Unknown / Not Reported
final_dataset_clean.loc[final_dataset_clean["race_raw_clean"].str.contains("unknown|refused", na=False),
       "race_standardized"] = "Unknown / Not Reported"

#revision de los resultados
print(final_dataset_clean['race_standardized'].value_counts())


Race
White                                        86439
Other                                        23175
White, non-Hispanic                          20179
Asian                                        17076
Unknown                                      13520
Black or African American                     8177
Race and Ethnicity Unknown                    7907
Other, Hispanic                               3237
Asian, non-Hispanic                           2518
Native Hawaiian or Other Pacific Islander     2186
Black, non-Hispanic                           1825
White, Hispanic                                861
Other, non-Hispanic                            521
American Indian or Alaska Native               383
Patient Refused                                342
Pacific Islander, non-Hispanic                 290
Native American, non-Hispanic                   51
Black, Hispanic                                 51
Asian, Hispanic                                 34
Native American, Hispanic 

/var/folders/9r/yqjc4kq94dgdmj6ny6g9_kjh0000gn/T/ipykernel_28650/3398265273.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_dataset_clean["race_standardized"] = np.nan
/var/folders/9r/yqjc4kq94dgdmj6ny6g9_kjh0000gn/T/ipykernel_28650/3398265273.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_dataset_clean["race_raw_clean"] = (final_dataset_clean["Race"].str.lower().str.strip())
/var/folders/9r/yqjc4kq94dgdmj6ny6g9_kjh0000gn/T/ipykernel_28650/3398265273.py:12: FutureWarning: Setting an ite

race_standardized
White                                        107479
Other                                         26933
Unknown / Not Reported                        21769
Asian                                         19649
Black or African American                     10053
Native Hawaiian or Other Pacific Islander      2486
American Indian or Alaska Native                459
Name: count, dtype: int64


In [68]:
# Revisión de la variable 'Ethnicity'
print(final_dataset_clean['Ethnicity'].value_counts())

# Estandarizacion de categorias de la variable 'Ethnicity'

final_dataset_clean["ethnicity_standardized"] = np.nan

# Limpiar texto
final_dataset_clean["ethnicity_clean"] = (
    final_dataset_clean["Ethnicity"]
    .str.lower()
    .str.strip()
)
# Hispanic / Latino
final_dataset_clean.loc[
    final_dataset_clean["ethnicity_clean"].str.contains("hispanic|latino", na=False),
    "ethnicity_standardized"
] = "Hispanic / Latino"


# Non-Hispanic 
final_dataset_clean.loc[
    final_dataset_clean["ethnicity_clean"].str.contains("non-hispanic|not hispanic", na=False),
    "ethnicity_standardized"
] = "Non-Hispanic"

# Unknown / Not Reported
final_dataset_clean.loc[
    final_dataset_clean["ethnicity_clean"].str.contains("unknown|patient refused", na=False),
    "ethnicity_standardized"
] = "Unknown / Not Reported"

# 4️⃣ Forzar Unknown si quedó algo sin clasificar
final_dataset_clean["ethnicity_standardized"] = (
    final_dataset_clean["ethnicity_standardized"]
    .fillna("Unknown / Not Reported")
)

#revision de los resultados
print(final_dataset_clean["ethnicity_standardized"].value_counts())


Ethnicity
Non-Hispanic/Non-Latino    138132
Unknown                     28249
Hispanic/Latino             22052
Patient Refused               381
Not Hispanic                   13
Hispanic                        1
Name: count, dtype: int64
ethnicity_standardized
Non-Hispanic              138145
Unknown / Not Reported     28630
Hispanic / Latino          22053
Name: count, dtype: int64


/var/folders/9r/yqjc4kq94dgdmj6ny6g9_kjh0000gn/T/ipykernel_28650/469120388.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_dataset_clean["ethnicity_standardized"] = np.nan
/var/folders/9r/yqjc4kq94dgdmj6ny6g9_kjh0000gn/T/ipykernel_28650/469120388.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_dataset_clean["ethnicity_clean"] = (
/var/folders/9r/yqjc4kq94dgdmj6ny6g9_kjh0000gn/T/ipykernel_28650/469120388.py:15: FutureWarning: Setting an item of incompatible dtype is deprecated and will ra

In [70]:
final_dataset_clean.columns
print(final_dataset_clean["Sex"].value_counts())

Sex
Male      110809
Female     78019
Name: count, dtype: int64
